# Contamination Check: ProtectAI DeBERTa Training Data vs Our Evaluation Datasets

Objective: check whether ProtectAI DeBERTa v3 base v2 was trained on prompts that overlap with our three evaluation datasets. Overlap would inflate the classifier's apparent performance.

**Method**: load each *named* training source, exact-string-match (lowercased, whitespace-normalized) against our eval prompts, count UNIQUE matches per dataset (union across sources in a tier, not sum).

**Two tiers of sources**:

**Tier 1: V2 named datasets (definitive)**. Named on the [V2 model card](https://huggingface.co/protectai/deberta-v3-base-prompt-injection-v2):
- `VMware/open-instruct`
- `alespalla/chatbot_instruction_prompts`
- `HuggingFaceH4/grok-conversation-harmless`
- `OpenSafetyLab/Salad-Data` (multi-config; we use `base_set`)
- `jackhhao/jailbreak-classification`
- `natolambert/xstest-v2-copy`

**Tier 2: V1 named datasets (precautionary)**. Named on the [V1 model card](https://huggingface.co/protectai/deberta-v3-base-prompt-injection), may or may not have been inherited into V2:
- `HuggingFaceH4/ultrachat_200k`
- `fka/prompts.chat`
- `HuggingFaceH4/no_robots`

**What we cannot check**:
- `Harelix/Prompt-Injection-Mixed-Techniques-2024`: original removed from HuggingFace. We tried `ahsanayub/malicious-prompts` as a recovery source per the [ShieldLM README](https://huggingface.co/datasets/Abdennebi/shieldlm-prompt-injection/blob/main/README.md), but it is a large aggregation that re-bundles deepset and SPML themselves, so overlap counts against it are uninterpretable. Harelix is therefore unverifiable.
- 15 additional V2 datasets disclosed only by license category (8 MIT, 1 CC0, 6 public-domain). ProtectAI's maintainer confirmed in an April 2024 HuggingFace Discussion that these details are kept confidential by design.
- Meta Llama Prompt Guard 2 has zero named training sources.

Use Python environment `.venv` (uv).


In [1]:
import os
import json
import pandas as pd
from pathlib import Path
from datasets import load_dataset, load_from_disk
from dotenv import load_dotenv

# resolve repo root
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".git").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / ".git").exists()

load_dotenv(dotenv_path=REPO_ROOT / ".env")

# report which known tokens loaded (presence only, no values), so you can confirm at a glance
print("Token status:")
for t in ["HF_TOKEN", "GROQ_API_KEY", "ANTHROPIC_API_KEY", "OPENAI_API_KEY"]:
    print(f"  {t}: {'set' if os.getenv(t) else 'not set'}")

# this notebook only requires HF_TOKEN
assert os.getenv("HF_TOKEN"), "HF_TOKEN required"

DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 80)

Token status:
  HF_TOKEN: set
  GROQ_API_KEY: not set
  ANTHROPIC_API_KEY: not set
  OPENAI_API_KEY: not set


## Load our evaluation prompts

Build a normalized set (lowercased, stripped) per dataset so we can intersect cheaply against each training source.

In [2]:
def normalize(s):
    return str(s).strip().lower()

df_deepset = load_from_disk(DATA_DIR / "deepset")["train"].to_pandas()
df_neuro   = load_from_disk(DATA_DIR / "neuralchemy")["train"].to_pandas()
df_spml    = load_from_disk(DATA_DIR / "spml")["train"].to_pandas()

our_prompts = {
    "deepset":     set(df_deepset["text"].map(normalize)),
    "neuralchemy": set(df_neuro["text"].map(normalize)),
    "spml":        set(df_spml["User Prompt"].map(normalize)),
}

for name, s in our_prompts.items():
    print(f"{name}: {len(s)} unique prompts after normalization")

deepset: 546 unique prompts after normalization
neuralchemy: 4391 unique prompts after normalization
spml: 15914 unique prompts after normalization


## Helper: download a source, extract prompts, compute overlap

Each HuggingFace dataset has different column structure. The helper tries common text column names (`text`, `prompt`, `instruction`, `user_prompt`, `question`). If none fit, it prints the schema so we can adjust.

In [ ]:
CANDIDATE_COLS = ["text", "prompt", "instruction", "user_prompt", "question", "input", "content"]

def extract_prompts(ds):
    """Extract normalized prompt strings from a HF dataset, trying candidate column names across all splits."""
    prompts = set()
    for split_name, split in ds.items() if hasattr(ds, "items") else [("train", ds)]:
        cols = split.column_names
        found = [c for c in CANDIDATE_COLS if c in cols]
        if not found:
            print(f"  [WARN] split={split_name}: no candidate column in {cols}")
            continue
        col = found[0]
        print(f"  split={split_name}: using column '{col}', {len(split)} rows")
        prompts.update(normalize(x) for x in split[col] if x is not None)
    return prompts

def check_overlap(source_id, config=None):
    """Download a source, extract prompts, compute overlap with each of our eval datasets.
    Returns a dict with counts PLUS the actual matched prompt sets (keys ending in '_matched_prompts'),
    so downstream union-across-sources totals don't double-count prompts matched in multiple sources."""
    print(f"\n=== {source_id}{f' ({config})' if config else ''} ===")
    try:
        ds = load_dataset(source_id, config) if config else load_dataset(source_id)
    except Exception as e:
        print(f"  [ERROR] load failed: {e}")
        return {"source": source_id, "error": str(e)}
    train_prompts = extract_prompts(ds)
    print(f"  total unique prompts extracted: {len(train_prompts)}")
    result = {"source": source_id, "training_prompts": len(train_prompts)}
    for name, ours in our_prompts.items():
        overlap = train_prompts & ours
        result[f"{name}_matches"] = len(overlap)
        result[f"{name}_matched_prompts"] = overlap  # private; dropped from CSV and display
        if overlap:
            sample = list(overlap)[:3]
            print(f"  {name}: {len(overlap)} exact matches. Samples:")
            for s in sample:
                print(f"    - {s[:100]}")
        else:
            print(f"  {name}: 0 matches")
    return result

## Run check on each disclosed ProtectAI training source

In [ ]:
sources = [
    # Tier 1: V2 named datasets (definitive). Harelix omitted (original removed from HF; no pure recovery).
    ("VMware/open-instruct",                          None,        "v2-named"),
    ("alespalla/chatbot_instruction_prompts",         None,        "v2-named"),
    ("HuggingFaceH4/grok-conversation-harmless",      None,        "v2-named"),
    ("OpenSafetyLab/Salad-Data",                      "base_set",  "v2-named"),
    ("jackhhao/jailbreak-classification",             None,        "v2-named"),
    ("natolambert/xstest-v2-copy",                    None,        "v2-named"),
    # Tier 2: V1 named datasets (precautionary; may have been inherited into V2)
    ("HuggingFaceH4/ultrachat_200k",                  None,        "v1-precaution"),
    ("fka/prompts.chat",                              None,        "v1-precaution"),
    ("HuggingFaceH4/no_robots",                       None,        "v1-precaution"),
]

results = []
for source_id, config, tier in sources:
    r = check_overlap(source_id, config)
    r["tier"] = tier
    results.append(r)

# display version drops the private match-set columns (keeps only counts)
df_results = pd.DataFrame(results)
display_cols = [c for c in df_results.columns if not c.endswith("_matched_prompts")]
df_results_display = df_results[display_cols]
df_results_display

## Save results

Writes a summary CSV and a brief markdown report. The markdown report is the one committed and referenced downstream.

In [ ]:
from datetime import datetime

# CSV keeps only count columns (no private match-set columns)
df_results_display.to_csv(RESULTS_DIR / "contamination_check.csv", index=False)

def tier_union_totals(results, tier_label):
    """Union of matched prompts across sources in a tier. A prompt matched in multiple sources counts once."""
    matched = {name: set() for name in our_prompts}
    for r in results:
        if r.get("tier") != tier_label or "error" in r:
            continue
        for name in our_prompts:
            matched[name].update(r.get(f"{name}_matched_prompts", set()))
    return {name: len(s) for name, s in matched.items()}

totals_v2 = tier_union_totals(results, "v2-named")
totals_v1 = tier_union_totals(results, "v1-precaution")

print("Tier 1 (V2 named, definitive) - UNIQUE prompts matched in at least one source:")
for name, total in totals_v2.items():
    print(f"  {name}: {total} / {len(our_prompts[name])} ({100*total/len(our_prompts[name]):.2f}%)")

print("\nTier 2 (V1 named, precautionary) - UNIQUE prompts matched in at least one source:")
for name, total in totals_v1.items():
    print(f"  {name}: {total} / {len(our_prompts[name])} ({100*total/len(our_prompts[name]):.2f}%)")

# write results/contamination_report.md
report_path = RESULTS_DIR / "contamination_report.md"
run_date = datetime.now().strftime("%Y-%m-%d")
lines = []
lines.append("# Classifier Contamination Report\n")
lines.append(f"*Run date: {run_date}. HuggingFace dataset IDs are stable, but datasets can be updated or removed by their owners; re-running at a later date may produce different results.*\n")
lines.append("Exact-string-match check between our three evaluation datasets and the *named* training sources of ProtectAI DeBERTa v3 base v2, plus the V1 model's named sources as a precautionary second tier. Meta Prompt Guard 2 is not checked (no source enumeration).\n")
lines.append("Totals are counted as UNIQUE prompts matched in at least one source within a tier (a prompt matched in multiple sources counts once per tier).\n")

lines.append("## Tier 1: V2 named sources (definitive)\n")
lines.append("Six of the seven V2-named datasets checked. Harelix is omitted because the original was removed from HuggingFace and no pure-recovery substitute was found (see limitations below).\n")
lines.append(df_results_display[df_results_display["tier"] == "v2-named"].drop(columns=["tier"]).to_markdown(index=False) + "\n")
lines.append("**Tier 1 totals (unique prompts matched)**:\n")
for name, total in totals_v2.items():
    pct = 100 * total / len(our_prompts[name])
    lines.append(f"- **{name}**: {total} of {len(our_prompts[name])} prompts matched ({pct:.2f}%)")

lines.append("\n## Tier 2: V1 named sources (precautionary)\n")
lines.append("Three datasets named on the V1 model card. V2 was retrained and its card does not enumerate V1's sources, so these may or may not have been inherited. Reported separately; treat as conservative upper-bound contamination.\n")
lines.append(df_results_display[df_results_display["tier"] == "v1-precaution"].drop(columns=["tier"]).to_markdown(index=False) + "\n")
lines.append("**Tier 2 totals (unique prompts matched)**:\n")
for name, total in totals_v1.items():
    pct = 100 * total / len(our_prompts[name])
    lines.append(f"- **{name}**: {total} of {len(our_prompts[name])} prompts matched ({pct:.2f}%)")

lines.append("\n## Unverifiable contamination risks (documented limitations)\n")
lines.append("### Harelix (original removed from HuggingFace)\n")
lines.append("`Harelix/Prompt-Injection-Mixed-Techniques-2024` is named on ProtectAI's V2 card but has been removed from HuggingFace. The [ShieldLM README](https://huggingface.co/datasets/Abdennebi/shieldlm-prompt-injection/blob/main/README.md) points at `ahsanayub/malicious-prompts` as a recovery source, but that dataset is a large aggregation of many prompt-injection datasets (including our deepset and SPML eval sets themselves), so overlap counts from it are uninterpretable for this check. Contamination from the original Harelix cannot be verified.\n")
lines.append("### ProtectAI V2 unnamed sources\n")
lines.append("15 additional V2 training datasets are disclosed by license category only (8 MIT, 1 CC0 1.0, 6 no-license/public-domain). ProtectAI's maintainer confirmed in an April 2024 [HuggingFace Discussion](https://huggingface.co/protectai/deberta-v3-base-prompt-injection-v2/discussions/1) that training-data details are kept partly confidential by design, and a follow-up asking to open-source the training data went unanswered.\n")
lines.append("### Meta Prompt Guard 2\n")
lines.append("The Meta Llama Prompt Guard 2 86M model card states only: 'a mix of open-source datasets reflecting benign data from the web, user prompts and instructions for LLMs, and malicious prompt injection and jailbreaking datasets... plus synthetic injections and red-teaming material.' No specific sources enumerated; no research paper expands on this.\n")

lines.append("## Handling decision\n")
lines.append("**Decision: accept and caveat in results.** Rationale: named-source overlap is small (at most 1.96% for any single eval dataset, and mostly well under 1%). Removing matched rows would prune the eval set by less than 100 prompts total across all three datasets and would not materially change Defense A metrics. Reporting contaminated vs clean metrics separately is overkill for overlap at this scale. The handling approach is therefore: leave the eval set intact, report the contamination numbers above in the methodology section of the final report, and note the unverifiable risks (Harelix, V2 unnamed sources, Meta) as explicit limitations. If Defense A's apparent performance on any specific neuralchemy subcategory looks suspiciously high during analysis, revisit this decision and re-examine whether overlap on that subcategory is concentrated enough to warrant removal.\n")
report_path.write_text("\n".join(lines), encoding="utf-8")
print(f"\nReport written to {report_path.relative_to(REPO_ROOT)}")